**Purpose of this Notebook**
1. Convert timestamp
2. Remove duplicate values
3. Enrich transactions with dimensions

In [0]:
RAW_PATH='/Volumes/workspace/default/eccomerce/'
BRONZE_PATH='/Volumes/workspace/default/eccomerce/bronze/'
SILVER_PATH='/Volumes/workspace/default/eccomerce/silver/'
GOLD_PATH='/Volumes/workspace/default/eccomerce/gold/'

#KEY OF GCP PROJECT
GCP_PROJECT='decisive-cinema-489218-g0'
BQ_QUERY='ecommerce'
TEMP_GCS_BUCKET='ecommerce-databricks-temp'


#secrets codes
GCP_SECRET_SCOPE='gcp-secret'
GCP_SECRET_KEY='gcp-sa-key'

In [0]:
from pyspark.sql.functions import to_timestamp, coalesce, lit, when, date_format, current_timestamp, col
from pyspark.sql.types import DoubleType

txn=spark.read.format('delta').load(BRONZE_PATH+'transactions')
cust=spark.read.format('delta').load(BRONZE_PATH+'customers')
prod=spark.read.format('delta').load(BRONZE_PATH+'products')
fb=spark.read.format('delta').load(BRONZE_PATH+'feedbacks')
store=spark.read.format('delta').load(BRONZE_PATH+'stores')
promo=spark.read.format('delta').load(BRONZE_PATH+'promotions')


cleaning data
1. remove duplicates from transaction id
2. cast total_amount into double type
3. date to print in date, month, year

In [0]:
txn_clean=(txn
           .dropDuplicates(['transaction_id'])
           .withColumn('total_amount', col('total_amount').cast(DoubleType()))
           .withColumn('transaction_date', date_format('transaction_date', 'dd-MM-yyyy'))
           .withColumn('quantity', col('quantity').cast('int'))
           .withColumn('total_amount', coalesce(col('total_amount'), lit(0.0)))
           .withColumn('_ingest_time', current_timestamp())
           )
txn_clean.display()

transaction_id,customer_id,product_id,store_id,transaction_date,quantity,total_amount,_ingest_time
12,14,10,2,04-10-2025,3,733.41,2026-03-07T09:27:11.788Z
18,11,14,5,04-10-2025,8,969.84,2026-03-07T09:27:11.788Z
16,10,9,1,02-10-2025,1,609.9,2026-03-07T09:27:11.788Z
5,12,4,1,02-10-2025,10,782.15,2026-03-07T09:27:11.788Z
10,12,7,5,03-10-2025,7,342.72,2026-03-07T09:27:11.788Z
1,9,6,3,01-10-2025,1,647.94,2026-03-07T09:27:11.788Z
3,9,2,2,03-10-2025,2,872.56,2026-03-07T09:27:11.788Z
20,13,14,5,03-10-2025,7,572.25,2026-03-07T09:27:11.788Z
2,7,19,5,01-10-2025,5,59.88,2026-03-07T09:27:11.788Z
13,9,10,1,04-10-2025,8,167.42,2026-03-07T09:27:11.788Z


In [0]:
silver=(txn_clean.alias('t')
        .join(cust.alias('c'), col('t.customer_id') == col('c.cust_id'), 'left')
        .join(prod.alias('p'), 'product_id', 'left')
        .join(store.alias('s'), 'store_id', 'left')
        .join(promo.alias('pr'), 'product_id', 'left')
        .join(fb.alias('f'), (col('t.customer_id') == col('f.customer_id')) & (col('t.product_id') == col('f.pproduct_id')), 'left')
        )

silver=(silver
        .withColumn('transaction_date',date_format(to_timestamp(col('transaction_date'), 'dd-MM-yyyy'),'yyyy-MM-dd'))
        .withColumn('is_valid', when(col('location').isNotNull(), lit(True)).otherwise(lit(False)))
        .withColumn('has_valid_customer', when(col('customer_name').isNotNull(), lit(True)).otherwise(lit(False)))
        .withColumn('final_amount', when(col('discount_percent').isNotNull(), col('total_amount')*(1-(col('discount_percent')/100))).otherwise(col('total_amount')))
        )

Processing Data:- data quality check and save data

In [0]:
total=silver.count()
invalid_store_count=silver.filter(~col('is_valid')).count()
missing_customer_count=silver.filter(~col('has_valid_customer')).count()

print(f"total rows:{total}, invalid store:{invalid_store_count}, missing customer:{missing_customer_count}")

if total>0 and (invalid_store_count /total)>0.10:
  raise Exception("DQ check failed:> 10% invalid store")


total rows:25, invalid store:0, missing customer:0


In [0]:
# Drop duplicate customer_id from feedbacks join
silver_clean = silver.drop(col('f.customer_id'))
(silver_clean.write.mode('overwrite').partitionBy('transaction_date').format('delta').save(SILVER_PATH+'transactions_enriched'))

In [0]:
from pyspark.sql.functions import to_date

promo=promo.withColumn('promo_start_date', to_date(col('promo_start_date'), 'yyyy-MM-dd'))\
    .withColumn('promo_end_date', to_date(col('promo_end_date'), 'yyyy-MM-dd'))

silver=silver.withColumn('transaction_date',to_date(col('transaction_date')))

silver=silver.withColumn('promo_in_range', when((col('discount_percent').isNotNull()) & (col('transaction_date') >= col('promo_start_date')) & (col('transaction_date') <= col('promo_end_date')), True).otherwise(False))

silver.display()

product_id,store_id,transaction_id,customer_id,transaction_date,quantity,total_amount,_ingest_time,cust_id,customer_name,age,gender,region,singup_date,product_name,category,price,brand,store_name,location,manager,open_date,promotion_id,discount_percent,promo_start_date,promo_end_date,channel,feedback_id,customer_id,pproduct_id,rating,review,review_date,is_valid,has_valid_customer,final_amount,promo_in_range
10,2,12,14,2025-10-04,3,733.41,2026-03-07T09:27:35.025Z,14,Anjali Joshi,22,Female,East,2023-08-14T00:00:00.000Z,External Hard Drive,Storage,5000.0,Seagate,GadgetHub,Pune,manager_2,2019-07-15T00:00:00.000Z,null,null,null,null,null,null,null,null,null,null,null,true,true,733.41,false
14,5,18,11,2025-10-04,8,969.84,2026-03-07T09:27:35.025Z,11,Sanjay Kulkarni,40,Male,West,2023-07-01T00:00:00.000Z,null,null,null,null,StyleStop,Chennai,manager_5,2022-12-01T00:00:00.000Z,null,null,null,null,null,null,null,null,null,null,null,true,true,969.84,false
9,1,16,10,2025-10-02,1,609.9,2026-03-07T09:27:35.025Z,10,Pooja Das,26,Female,South,2023-06-12T00:00:00.000Z,Printer,Electronics,8000.0,Canon,TechWorld,Mumbai,manager_1,2018-05-10T00:00:00.000Z,9,31.0,2025-05-01T00:00:00.000Z,2025-05-10T00:00:00.000Z,Online,null,null,null,null,null,null,true,true,420.83099999999996,false
4,1,5,12,2025-10-02,10,782.15,2026-03-07T09:27:35.025Z,12,Deepa Menon,33,Female,North,2023-07-20T00:00:00.000Z,Smartwatch,Wearable,15000.0,Fitbit,TechWorld,Mumbai,manager_1,2018-05-10T00:00:00.000Z,null,null,null,null,null,null,null,null,null,null,null,true,true,782.15,false
7,5,10,12,2025-10-03,7,342.72,2026-03-07T09:27:35.025Z,12,Deepa Menon,33,Female,North,2023-07-20T00:00:00.000Z,Mouse,Accessories,800.0,Dell,StyleStop,Chennai,manager_5,2022-12-01T00:00:00.000Z,10,26.0,2025-05-15T00:00:00.000Z,2025-05-30T00:00:00.000Z,App,null,null,null,null,null,null,true,true,253.61280000000002,false
6,3,1,9,2025-10-01,1,647.94,2026-03-07T09:27:35.025Z,9,Vikas Nair,31,Male,East,2023-06-01T00:00:00.000Z,Keyboard,Accessories,1200.0,Logitech,BookNest,Delhi,manager_3,2020-03-20T00:00:00.000Z,6,29.0,2025-03-15T00:00:00.000Z,2025-03-30T00:00:00.000Z,Online,null,null,null,null,null,null,true,true,460.0374,false
2,2,3,9,2025-10-03,2,872.56,2026-03-07T09:27:35.025Z,9,Vikas Nair,31,Male,East,2023-06-01T00:00:00.000Z,Smartphone,Electronics,45000.0,Samsung,GadgetHub,Pune,manager_2,2019-07-15T00:00:00.000Z,5,25.0,2025-03-01T00:00:00.000Z,2025-03-10T00:00:00.000Z,Retail,null,null,null,null,null,null,true,true,654.42,false
14,5,20,13,2025-10-03,7,572.25,2026-03-07T09:27:35.025Z,13,Manish Gupta,38,Male,South,2023-08-01T00:00:00.000Z,null,null,null,null,StyleStop,Chennai,manager_5,2022-12-01T00:00:00.000Z,null,null,null,null,null,null,null,null,null,null,null,true,true,572.25,false
19,5,2,7,2025-10-01,5,59.88,2026-03-07T09:27:35.025Z,7,Rahul Verma,35,Male,West,2023-05-02T00:00:00.000Z,null,null,null,null,StyleStop,Chennai,manager_5,2022-12-01T00:00:00.000Z,null,null,null,null,null,null,null,null,null,null,null,true,true,59.88,false
10,1,13,9,2025-10-04,8,167.42,2026-03-07T09:27:35.025Z,9,Vikas Nair,31,Male,East,2023-06-01T00:00:00.000Z,External Hard Drive,Storage,5000.0,Seagate,TechWorld,Mumbai,manager_1,2018-05-10T00:00:00.000Z,null,null,null,null,null,null,null,null,null,null,null,true,true,167.42,false
